In [1]:
"""
fetch_data.py

Download historical weather data from NASA POWER and estimate daily
solar and wind energy generation.

Data source:
https://power.larc.nasa.gov/
"""

import requests
import pandas as pd
import numpy as np


# Configuration

LATITUDE = 33.5779
LONGITUDE = -101.8313

START = "20180101"
END = "20241231"


# Solar plant parameters

PV_CAPACITY_MW = 50.0
PV_PERFORMANCE_RATIO = 0.80
PV_TEMP_COEFF = -0.004
PV_REFERENCE_TEMP = 25.0
PV_NOCT = 45.0


# Wind plant parameters

WIND_CAPACITY_MW = 50.0
WIND_CUT_IN = 3.0
WIND_RATED = 12.0
WIND_CUT_OUT = 25.0


def fetch_nasa_power(lat, lon, start, end):
    """Download daily weather data from NASA POWER."""

    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        "?parameters="
        "ALLSKY_SFC_SW_DWN,"
        "CLRSKY_SFC_SW_DWN,"
        "WS10M,"
        "WS50M,"
        "T2M,"
        "RH2M,"
        "PRECTOTCORR"
        "&community=RE"
        f"&longitude={lon}"
        f"&latitude={lat}"
        f"&start={start}"
        f"&end={end}"
        "&format=JSON"
    )

    print("Requesting NASA POWER data...")

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    data = response.json()
    parameters = data["properties"]["parameter"]

    df = pd.DataFrame(parameters)

    df.index = pd.to_datetime(
        df.index.astype(str),
        format="%Y%m%d"
    )

    df.index.name = "date"

    # NASA POWER uses -999 for missing values.
    df = df.replace(-999, np.nan)

    return df


def solar_energy_mwh(
    solar_irradiation_kwh_m2_day,
    ambient_temp_c,
    capacity_mw=PV_CAPACITY_MW
):
    """Estimate daily solar energy generation in MWh."""

    # Estimate PV cell temperature using a simplified NOCT model.
    cell_temp = (
        ambient_temp_c
        + ((PV_NOCT - 20.0) / 800.0)
        * (solar_irradiation_kwh_m2_day * 1000.0 / 24.0)
    )

    # Apply temperature derating.
    temp_factor = (
        1.0
        + PV_TEMP_COEFF
        * (cell_temp - PV_REFERENCE_TEMP)
    )

    temp_factor = temp_factor.clip(lower=0)

    energy = (
        solar_irradiation_kwh_m2_day
        * capacity_mw
        * PV_PERFORMANCE_RATIO
        * temp_factor
    )

    return energy.clip(lower=0)


def wind_power_mw(
    wind_speed_ms,
    capacity_mw=WIND_CAPACITY_MW
):
    """Estimate wind power using a simplified turbine power curve."""

    wind_speed_ms = pd.Series(
        wind_speed_ms,
        index=wind_speed_ms.index
    )

    power = pd.Series(
        0.0,
        index=wind_speed_ms.index
    )

    # Cubic power region between cut-in and rated speed.
    ramp = (
        (wind_speed_ms >= WIND_CUT_IN)
        & (wind_speed_ms < WIND_RATED)
    )

    fraction = (
        (wind_speed_ms - WIND_CUT_IN)
        / (WIND_RATED - WIND_CUT_IN)
    ) ** 3

    power.loc[ramp] = (
        capacity_mw
        * fraction.loc[ramp]
    )

    # Rated power region.
    rated = (
        (wind_speed_ms >= WIND_RATED)
        & (wind_speed_ms < WIND_CUT_OUT)
    )

    power.loc[rated] = capacity_mw

    return power.clip(
        lower=0,
        upper=capacity_mw
    )


def main():
    """Download data, estimate generation and save the dataset."""

    df = fetch_nasa_power(
        LATITUDE,
        LONGITUDE,
        START,
        END
    )

    print("\nData downloaded successfully.")
    print(f"Number of rows: {len(df)}")

    # Fill short gaps and remove remaining missing rows.
    df = df.interpolate(
        method="linear",
        limit=3
    )

    df = df.dropna()

    # Estimate solar generation.
    df["solar_energy_mwh"] = solar_energy_mwh(
        df["ALLSKY_SFC_SW_DWN"],
        df["T2M"]
    )

    # Use 50 m wind speed when available.
    if "WS50M" in df.columns:
        wind_column = "WS50M"
    else:
        wind_column = "WS10M"

    df["wind_power_mw"] = wind_power_mw(
        df[wind_column]
    )

    df["wind_energy_mwh"] = (
        df["wind_power_mw"] * 24.0
    )

    # Combined renewable generation.
    df["total_energy_mwh"] = (
        df["solar_energy_mwh"]
        + df["wind_energy_mwh"]
    )

    # Time-based features.
    df["day_of_year"] = df.index.dayofyear
    df["month"] = df.index.month
    df["year"] = df.index.year
    df["day_of_week"] = df.index.dayofweek

    # Save dataset.
    df.to_csv("generation_data.csv")

    print("\nDataset successfully created!")
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")

    print("\nLast 5 rows:")

    print(
        df[
            [
                "ALLSKY_SFC_SW_DWN",
                wind_column,
                "T2M",
                "solar_energy_mwh",
                "wind_energy_mwh",
                "total_energy_mwh"
            ]
        ].tail()
    )

    print("\nSaved as: generation_data.csv")


if __name__ == "__main__":
    main()



Requesting NASA POWER data...

Data downloaded successfully.
Number of rows: 2557

Dataset successfully created!
Rows: 2557
Columns: 15

Last 5 rows:
            ALLSKY_SFC_SW_DWN  WS50M    T2M  solar_energy_mwh  \
date                                                            
2024-12-27             2.5968   9.47  10.76        108.383680   
2024-12-28             3.5052   7.31  10.36        145.858908   
2024-12-29             3.5042   7.48  11.66        145.089152   
2024-12-30             3.5551  11.46  13.15        146.311400   
2024-12-31             3.5443   5.89   4.91        150.547702   

            wind_energy_mwh  total_energy_mwh  
date                                           
2024-12-27       445.827198        554.210879  
2024-12-28       131.790932        277.649840  
2024-12-29       148.008876        293.098028  
2024-12-30       996.700800       1143.012200  
2024-12-31        39.732624        190.280325  

Saved as: generation_data.csv
